In [1]:
import json

import pandas as pd
import ast
import os
from tqdm import tqdm

from typing import Union, List

In [2]:
tqdm.pandas()

# Load Impact DF

In [3]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [4]:
dataset_dir = "/home/yishin/keith/patent_research/model_io"

In [6]:
# Impact_df = pd.read_csv(os.path.join(dataset_dir, "0_Base.csv"), encoding="utf-8")
Impact_df = pd.read_csv(os.path.join(dataset_dir, "0_Impact.csv"), encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,caption,image_paths,fig_desc,Loc_class,main_class,sub_class,first_class,best_fig_desc
0,Combination card and key holder,The image is a white outline of a combination ...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,{03-01},3,1,03-01,{'FIG. 1': 'FIG. 1 is a perspective view of th...
1,Container,"The image is a white drawing of a container, w...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...",{09-01},9,1,09-01,"{'FIG. 1': 'FIG. 1 is a top, front, right side..."
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a ...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,{14-04},14,4,14-04,{}
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a ...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,{14-04},14,4,14-04,{'FIG. 1': 'FIG. 1 is a front view of a mobile...
4,Bottle cap opener,"The image is a drawing of a bottle cap opener,...",[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,"{07-99, 07-06}",7,99,07-99,{'FIG. 7': 'FIG. 7 is a perspective view there...


# Preprocessing

## USPC/Locarno

### USPC Normalization

In [6]:
with open("class_folder/USPC_RANGES.json", "r", encoding="utf-8") as f:
    USPC_RANGES = json.load(f)

In [7]:
def normalize_code(code: Union[str, List[str]]) -> set[str]:
    results = set()

    # ---- normalize input into a list of strings ----
    if isinstance(code, str):
        codes_to_process = code.replace(" ", "").split(",")
    else:
        # list of strings → clean each, split commas if present
        codes_to_process = []
        for c in code:
            if isinstance(c, str):
                codes_to_process.extend(c.replace(" ", "").split(","))

    # ---- core normalization logic ----
    for single_code in codes_to_process:
        for prefix_len in (2, 3):
            if len(single_code) <= prefix_len:
                continue

            prefix = single_code[:prefix_len]
            suffix = single_code[prefix_len:]

            # reject leading-zero suffixes
            if len(suffix) > 1 and suffix.startswith("0"):
                continue

            ranges = USPC_RANGES.get(prefix)
            if not ranges:
                continue

            try:
                suffix_int = int(suffix)
            except ValueError:
                continue

            start, end = ranges
            if start <= suffix_int < end:
                results.add(f"{prefix}-{suffix}")

    return results

In [8]:
Impact_df["USPC_class"] = Impact_df["class"].apply(normalize_code)

Impact_df.head()

,title,caption,file_names,fig_desc,class,USPC_class
0,Combination card and key holder,The image is a white outline of a combination ...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,D 3208,{D3-208}
1,Container,"The image is a white drawing of a container, w...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...","D 9542, D9574","{D9-574, D9-542}"
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a ...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,D14486,{D14-486}
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a ...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,D14486,{D14-486}
4,Bottle cap opener,"The image is a drawing of a bottle cap opener,...",[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,D 8 40,{D8-40}


### USPC to Locarno Conversion

In [9]:
with open("class_folder/USPC_CONVERSION_CHART.json", "r", encoding="utf-8") as f:
    CONVERSION_CHART = json.load(f)

In [10]:
def convert_USPC_Locarno(USPC_Codes: set):
    Locarno_Codes = []

    for i in USPC_Codes:
        try:
            main_class, subclass = i.split("-")
        except ValueError:
            return set()
        
        for cat in CONVERSION_CHART[main_class]:
            bound = cat["U.S. Subclass"].replace(" ", "").split("-")

            if len(bound) == 1:
                if subclass == bound[0]:
                    Locarno_Codes.append(cat["Locarno Class - Subclass"].replace(" ", ""))
            else:
                if float(bound[0]) <= float(subclass) <= float(bound[1]):
                    Locarno_Codes.append(cat["Locarno Class - Subclass"].replace(" ", ""))

    return set(Locarno_Codes)

In [11]:
Impact_df["Loc_class"] = Impact_df["USPC_class"].apply(convert_USPC_Locarno)

Impact_df.head()

,title,caption,file_names,fig_desc,class,USPC_class,Loc_class
0,Combination card and key holder,The image is a white outline of a combination ...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,D 3208,{D3-208},{03-01}
1,Container,"The image is a white drawing of a container, w...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...","D 9542, D9574","{D9-574, D9-542}",{09-01}
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a ...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,D14486,{D14-486},{14-04}
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a ...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,D14486,{D14-486},{14-04}
4,Bottle cap opener,"The image is a drawing of a bottle cap opener,...",[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,D 8 40,{D8-40},"{07-99, 07-06}"


### Drop USPC

In [12]:
Impact_df = Impact_df.drop(columns=["class"])
Impact_df = Impact_df.drop(columns=["USPC_class"])

### Empty Check

In [13]:
def is_class_empty(x):
    return (
        pd.isna(x)
        or isinstance(x, str)
        or (isinstance(x, set) and len(x) == 0)
    )

In [14]:
mask = (
    Impact_df["Loc_class"].apply(is_class_empty)
)
Impact_df = Impact_df[~mask]

In [15]:
def to_set_or_empty(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, (list, tuple, set)):
        return set(x)
    return {str(x)}

In [16]:
row_has_empty_mask = Impact_df.apply(
    lambda row: any(len(to_set_or_empty(v)) == 0 for v in row),
    axis=1,
)

In [17]:
row_has_empty_mask.sum()
Impact_df = Impact_df[~row_has_empty_mask].reset_index(drop=True)
Impact_df.shape[0]

387161

In [18]:
Impact_df.head()

,title,caption,file_names,fig_desc,Loc_class
0,Combination card and key holder,The image is a white outline of a combination ...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,{03-01}
1,Container,"The image is a white drawing of a container, w...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...",{09-01}
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a ...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,{14-04}
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a ...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,{14-04}
4,Bottle cap opener,"The image is a drawing of a bottle cap opener,...",[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,"{07-99, 07-06}"


## Class Dissection

In [22]:
def split_class(value):
    if not value:
        return pd.Series([None, None])

    first_item = list(value)[0]

    main_class, sub_class = first_item.split("-")

    return pd.Series([first_item, main_class, sub_class])

In [28]:
Impact_df[["first_class","main_class", "sub_class"]] = (
    Impact_df["Loc_class"].apply(split_class)
)

In [29]:
Impact_df.head(10)

,title,caption,file_names,fig_desc,Loc_class,main_class,sub_class,first_class
0,Combination card and key holder,The image is a white outline of a combination ...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,{03-01},03,01,03-01
1,Container,"The image is a white drawing of a container, w...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...",{09-01},09,01,09-01
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a ...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,{14-04},14,04,14-04
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a ...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,{14-04},14,04,14-04
4,Bottle cap opener,"The image is a drawing of a bottle cap opener,...",[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,"{07-99, 07-06}",07,99,07-99
5,Combination controlled cash bag lock,The image is a black and white drawing of a Co...,[impact_dataset/2013/USD0694610-20131203/USD06...,[FIG. 1 is a left end view of a combination co...,{08-07},08,07,08-07
6,Tip for a skin treatment apparatus,The image is a drawing of a Tip for a skin tre...,[impact_dataset/2013/USD0674485-20130115/USD06...,[This application is related to the United Sta...,{24-02},24,02,24-02
7,Rear bumper for an automobile,"The image is a rectangular shape, showcasing t...",[impact_dataset/2013/USD0695165-20131210/USD06...,[FIG. 1 is a perspective view of a rear bumper...,{12-16},12,16,12-16
8,Disposable apparatus for cell separation,The image is a white circle with a black outli...,[impact_dataset/2013/USD0694423-20131126/USD06...,[FIG. 1 is a perspective view of a disposable ...,{24-02},24,02,24-02
9,Loveseat,The image is a white outline of a wooden bench...,[impact_dataset/2013/USD0679109-20130402/USD06...,[FIG. 1 is a frontal perspective view of the l...,{06-01},06,01,06-01


## Image Paths and Fig Desc

In [30]:
from __future__ import annotations
 
import re
from typing import Optional

In [31]:
Impact_df = Impact_df.rename(columns={"file_names": "image_paths"})

Impact_df.head()

,title,caption,image_paths,fig_desc,Loc_class,main_class,sub_class,first_class
0,Combination card and key holder,The image is a white outline of a combination ...,[impact_dataset/2013/USD0680733-20130430/USD06...,[FIG. 1 is a perspective view of the combinati...,{03-01},03,01,03-01
1,Container,"The image is a white drawing of a container, w...",[impact_dataset/2013/USD0688951-20130903/USD06...,"[FIG. 1 is a top, front, right side perspectiv...",{09-01},09,01,09-01
2,Display screen with a graphical user interface,"The image is a square shape, and it depicts a ...",[impact_dataset/2013/USD0695303-20131210/USD06...,[The FIGURE is a view of a display screen with...,{14-04},14,04,14-04
3,Mobile phone with a graphical user interface,"The image is square-shaped, and it features a ...",[impact_dataset/2013/USD0696676-20131231/USD06...,[The patent or application file contains at le...,{14-04},14,04,14-04
4,Bottle cap opener,"The image is a drawing of a bottle cap opener,...",[impact_dataset/2013/USD0683603-20130604/USD06...,[FIG. 1 is a front view of a bottle cap opener...,"{07-99, 07-06}",07,99,07-99


In [32]:
_EXCLUDE_PHRASES = [
    r"reference view",
    r"state of use",
    r"state of conduction",
    r"in use",
    r"environment",
    r"prior art",
    r"exploded view",
    r"cross.?section",
    r"sectional view",
    r"detail view",
    r"enlarged view",
    r"schematic",
]

_EXCLUDE_RE = re.compile(
    "|".join(_EXCLUDE_PHRASES),
    re.IGNORECASE,
)

In [33]:
_TIER: list[tuple[int, re.Pattern]] = [
    # Perspective views capture three faces at once — highest value.
    (100, re.compile(r"perspective", re.IGNORECASE)),
 
    # Primary faces: front and rear.
    (70,  re.compile(r"\bfront\b", re.IGNORECASE)),
    (65,  re.compile(r"\brear\b",  re.IGNORECASE)),
 
    # Secondary faces: sides.
    (50,  re.compile(r"\b(left|right).?side\b", re.IGNORECASE)),
    (45,  re.compile(r"\bside\b",  re.IGNORECASE)),
 
    # Plan views — useful but often less distinctive.
    (30,  re.compile(r"\btop\b",    re.IGNORECASE)),
    (25,  re.compile(r"\bbottom\b", re.IGNORECASE)),
]

_MULTI_FACE_BONUS = 15   # added when a description mentions 2+ anatomical terms
_FACE_TERMS = re.compile(
    r"\b(front|rear|left|right|top|bottom|side)\b",
    re.IGNORECASE,
)

In [34]:
def _score(description: str) -> int:
    """Return an importance score for a single figure description."""
    best = 0
    for score, pattern in _TIER:
        if pattern.search(description):
            best = max(best, score)
 
    # Bonus when the view description mentions multiple spatial terms
    # (e.g. "front, left-side, top perspective").
    face_count = len(_FACE_TERMS.findall(description))
    if face_count >= 2:
        best += _MULTI_FACE_BONUS
 
    return best

def _parse_fig_number(description: str) -> int:
    """Extract the FIG number for stable tie-breaking (lower = earlier)."""
    match = re.search(r"\bFIG\.?\s*(\d+)", description, re.IGNORECASE)
    return int(match.group(1)) if match else 9999
 
 
def select_important_figures(
    descriptions: list[str],
    n: int = 4,
    exclude_re: Optional[re.Pattern] = None,
) -> list[dict]:
    """
    Select the *n* most informative figure descriptions.
 
    Parameters
    ----------
    descriptions : list[str]
        Raw figure-description strings, e.g. from the patent's brief
        description of drawings section.
 
    n : int
        Number of figures to return (default 4).
 
    exclude_re : re.Pattern, optional
        Override the default exclusion regex.  Pass ``None`` to use the
        built-in ``_EXCLUDE_RE``.
 
    Returns
    -------
    list[dict]
        Sorted by score descending, each entry::
 
            {
                "fig":         "FIG. 1",
                "description": "FIG. 1 is a front, left-side, top ...",
                "score":       115,
            }
 
    Examples
    --------
    >>> results = select_important_figures(descriptions, n=3)
    >>> for r in results:
    ...     print(r["fig"], r["score"], r["description"][:60])
    """
    excl = exclude_re if exclude_re is not None else _EXCLUDE_RE
 
    scored = []
    for desc in descriptions:
        if excl.search(desc):
            continue                        # skip reference / non-ornamental views
        score = _score(desc)
        if score == 0:
            continue                        # unrecognised / unlabelled — skip
        scored.append({
            "fig":         re.search(r"FIG\.?\s*\d+", desc, re.IGNORECASE).group() if re.search(r"FIG\.?\s*\d+", desc, re.IGNORECASE) else "?",
            "description": desc,
            "score":       score,
            "_fig_num":    _parse_fig_number(desc),
        })
 
    # Sort: score descending, then fig number ascending for stable tie-breaking.
    scored.sort(key=lambda x: (-x["score"], x["_fig_num"]))
 
    top = scored[:n]
 
    # Return without the internal sort key.
    return [{k: v for k, v in entry.items() if k != "_fig_num"} for top in [top] for entry in top]
 
 
# ---------------------------------------------------------------------------
# Convenience wrapper for use inside a DataFrame pipeline
# ---------------------------------------------------------------------------
 
def select_fig_indices(descriptions: list[str], n: int = 4) -> list[int]:
    """
    Return the 0-based indices into *descriptions* of the selected figures.
 
    Useful when you need to filter a parallel list of image paths::
 
        fig_descs  = row["fig_descriptions"]   # list[str]
        image_paths = row["image_paths"]        # list[str], same order
 
        indices     = select_fig_indices(fig_descs, n=4)
        top_paths   = [image_paths[i] for i in indices]
    """
    selected_descs = {
        entry["description"]
        for entry in select_important_figures(descriptions, n=n)
    }
    return [i for i, d in enumerate(descriptions) if d in selected_descs]

In [40]:
Impact_df["best_fig_desc"] = (
    Impact_df["fig_desc"]
    .map(try_literal_eval)
    .apply(lambda descs: {
        r["fig"]: r["description"]
        for r in select_important_figures(descs, n=5)
    })
)

In [41]:
Impact_df["best_fig_desc"].iloc[0]

{'FIG. 1': 'FIG. 1 is a perspective view of the combination card and key holder in accordance with the invention, the dashed lines forming no part of the claimed design;',
 'FIG. 6': 'FIG. 6 is a perspective view thereof, showing the key tags fanned out for viewing, the dashed lines forming no part of the claimed design; and,',
 'FIG. 3': 'FIG. 3 is a front elevation view thereof, the dashed lines forming no part of the claimed design;',
 'FIG. 7': 'FIG. 7 is a front elevation view thereof, showing the key tags fanned out for viewing, the dashed lines forming no part of the claimed design.',
 'FIG. 4': 'FIG. 4 is a left side elevation view thereof, the dashed lines forming no part of the claimed design;'}

In [43]:
Impact_df.to_csv("model_io/0_Impact.csv", index=False, encoding="utf-8")

# Subsampling

In [7]:
len(Impact_df)

387161

In [8]:
print(Impact_df.index.name)
print(Impact_df.columns.tolist())

None
['title', 'caption', 'image_paths', 'fig_desc', 'Loc_class', 'main_class', 'sub_class', 'first_class', 'best_fig_desc']


In [9]:
import numpy as np

def sample_group(group, alpha=0.5, scale=1, min_n=10):
    n = int(scale * (len(group) ** alpha))
    n = max(min_n, n)
    n = min(n, len(group))  # final cap must be group size
    return group.sample(n=n, random_state=42)

sample_df = (
    Impact_df
    .groupby("main_class", group_keys=False)
    .apply(lambda g: sample_group(g).assign(main_class=g.name))
    .reset_index(drop=True)
)

In [10]:
len(sample_df)

3076

In [11]:
sample_df.head()

,title,caption,image_paths,fig_desc,Loc_class,sub_class,first_class,best_fig_desc,main_class
0,Snack food product,"The image is a rectangular shape, and it is a ...",[impact_dataset/2017/USD0787150-20170523/USD07...,[FIG. 1 is a perspective view of a snack food ...,"{11-01, 01-01}",1,01-01,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,1
1,Ring-like input device,The image is a square-shaped drawing of a ring...,[impact_dataset/2013/USD0673953-20130108/USD06...,[FIG. 1 is a perspective view of a ring-like i...,"{11-01, 14-02, 01-01}",1,01-01,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,1
2,Snack food,The image is a black and white drawing of a sn...,[impact_dataset/2016/USD0752316-20160329/USD07...,"[FIG. 1 is a top view; and,, FIG. 2 is a persp...","{11-01, 01-01}",1,01-01,"{'FIG. 2': 'FIG. 2 is a perspective view.', 'F...",1
3,Food bar,"The image is square-shaped, and it is a food b...",[impact_dataset/2019/USD0862830-20191015/USD08...,[The patent or application file contains at le...,{01-01},1,01-01,{'FIG. 5': 'FIG. 5 is a front elevation view o...,1
4,Bracelet,"The image is a white outline of a bracelet, wh...",[impact_dataset/2018/USD0830218-20181009/USD08...,[FIG. 1 is a front perspective view of a brace...,"{11-01, 01-01}",1,01-01,{'FIG. 1': 'FIG. 1 is a front perspective view...,1


In [12]:
print(sample_df["main_class"].value_counts())

main_class
14    241
12    183
2     162
9     156
24    154
23    149
6     138
13    136
8     135
21    128
26    127
15    124
7     122
10    104
16     99
11     84
25     84
28     77
22     73
3      71
30     65
19     63
4      59
18     49
27     48
20     46
29     46
1      39
5      37
31     37
17     31
32      9
Name: count, dtype: int64


In [13]:
sample_df = sample_df[["title", "caption", "image_paths", "best_fig_desc", "Loc_class", "main_class", "sub_class", "first_class"]]

In [14]:
sample_df.head()

,title,caption,image_paths,best_fig_desc,Loc_class,main_class,sub_class,first_class
0,Snack food product,"The image is a rectangular shape, and it is a ...",[impact_dataset/2017/USD0787150-20170523/USD07...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{11-01, 01-01}",1,1,01-01
1,Ring-like input device,The image is a square-shaped drawing of a ring...,[impact_dataset/2013/USD0673953-20130108/USD06...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{11-01, 14-02, 01-01}",1,1,01-01
2,Snack food,The image is a black and white drawing of a sn...,[impact_dataset/2016/USD0752316-20160329/USD07...,"{'FIG. 2': 'FIG. 2 is a perspective view.', 'F...","{11-01, 01-01}",1,1,01-01
3,Food bar,"The image is square-shaped, and it is a food b...",[impact_dataset/2019/USD0862830-20191015/USD08...,{'FIG. 5': 'FIG. 5 is a front elevation view o...,{01-01},1,1,01-01
4,Bracelet,"The image is a white outline of a bracelet, wh...",[impact_dataset/2018/USD0830218-20181009/USD08...,{'FIG. 1': 'FIG. 1 is a front perspective view...,"{11-01, 01-01}",1,1,01-01


In [16]:
sample_df.to_csv("model_io/1_Impact_Sub.csv", index=False, encoding="utf-8")